<a href="https://colab.research.google.com/github/RIDDHI1624/Drug-Discovery/blob/main/Insulin_Receptor_Project/Phase3_gromacs_MD_Validation_step1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3.3 — GROMACS MD Validation · Step 1: System Preparation
## AI Drug Discovery Pipeline · Insulin Receptor (INSR)

### How this links to previous steps
- **Step 2 (GNINA)** → `phase3_step2_hits.sdf` — 5 docked poses
- **Step 3 (Pose Quality)** → RMSD ✓, strain energy ✓, fingerprint done
- **This step** → builds full MD simulation system around best GNINA pose

### Strategy
The ligand ITP requires proper GAFF2 atom types (needs AmberTools locally).
On Colab we run **protein-only solvation** to validate Steps 10, 12, 13.
Ligand is tracked separately and will be added in a local GAFF2 environment.

| Step | Task | Status on Colab |
|---|---|---|
| 10 | Protein topology (pdb2gmx) |  Full |
| 11 | Ligand parameterization (GAFF2) | ⚠ Geometry-only (no AmberTools) |
| 12 | Complex assembly |  Protein + ligand coordinates |
| 13 | Solvation (TIP3P + NaCl) |  Full |

---
### Upload before running
- `phase3_step2_hits.sdf`
- `synthetic_holo_template.pdb`

---
## 0 · Install GROMACS

In [1]:
!apt-get install -y gromacs openbabel -q
%pip install rdkit biopython pandas -q

import subprocess
r = subprocess.run(['gmx','--version'], capture_output=True, text=True)
ver = [l for l in r.stdout.split('\n') if 'GROMACS version' in l]
print(f'✓ {ver[0].strip() if ver else "GROMACS installed"}')

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  fonts-mathjax gromacs-data libboost-iostreams1.74.0 libfftw3-double3
  libfftw3-single3 libgromacs6 libinchi1 libjs-mathjax libmaeparser1
  libopenbabel7 sse4.2-support
Suggested packages:
  pymol libfftw3-bin libfftw3-dev fonts-mathjax-extras fonts-stix
  libjs-mathjax-doc
The following NEW packages will be installed:
  fonts-mathjax gromacs gromacs-data libboost-iostreams1.74.0 libfftw3-double3
  libfftw3-single3 libgromacs6 libinchi1 libjs-mathjax libmaeparser1
  libopenbabel7 openbabel sse4.2-support
0 upgraded, 13 newly installed, 0 to remove and 42 not upgraded.
Need to get 62.4 MB of archives.
After this operation, 329 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 sse4.2-support amd64 6 [8,572 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-mathjax all 2.7.9+dfsg-1 [2,208 

---
## 1 · Imports

In [2]:
import os, subprocess, shutil, zipfile
import numpy as np
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, SDWriter
from Bio.PDB import PDBParser
print('✓ Imports OK.')

✓ Imports OK.


---
## 2 · Configuration — Edit Only This Cell

In [3]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────
HITS_SDF    = Path('/content/phase3_step2_hits.sdf')
PROTEIN_PDB = Path('/content/synthetic_holo_template.pdb')
WORK_DIR    = Path('/content/gromacs_md')

FORCE_FIELD    = 'amber99sb-ildn'  # available in Colab GROMACS
WATER_MODEL    = 'tip3p'
BOX_DISTANCE   = 1.2    # nm = 12Å (pipeline spec)
IONIC_STRENGTH = 0.15   # M NaCl (pipeline spec)
BEST_CANDIDATE = 'candidate_0004'  # highest CNN score from Step 2
# ────────────────────────────────────────────────────────────────────────────
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f'Work dir       : {WORK_DIR}')
print(f'Force field    : {FORCE_FIELD}')
print(f'Box distance   : {BOX_DISTANCE} nm = {BOX_DISTANCE*10:.0f}Å')
print(f'Ionic strength : {IONIC_STRENGTH} M NaCl')
print(f'Best candidate : {BEST_CANDIDATE}')

Work dir       : /content/gromacs_md
Force field    : amber99sb-ildn
Box distance   : 1.2 nm = 12Å
Ionic strength : 0.15 M NaCl
Best candidate : candidate_0004


---
## 3 · Validate Inputs

In [4]:
errors = []
for path, label in [
    (HITS_SDF,    'GNINA hits SDF'),
    (PROTEIN_PDB, 'Protein PDB'),
]:
    if not path.exists():
        errors.append(f'✗ {label} missing: {path}')
    else:
        print(f'✓ {label}: {path.name} ({path.stat().st_size/1024:.0f} KB)')

if subprocess.run(['which','gmx'], capture_output=True).returncode != 0:
    errors.append('✗ GROMACS not found — re-run Cell 0')
else:
    print('✓ GROMACS: found')

if errors:
    for e in errors: print(e)
    raise RuntimeError('Upload missing files via left sidebar.')

# Load GNINA hits
hit_mols = [m for m in Chem.SDMolSupplier(str(HITS_SDF), removeHs=False)
            if m is not None]
print(f'\nGNINA hit molecules: {len(hit_mols)}')
for m in hit_mols:
    name  = m.GetProp('_Name')          if m.HasProp('_Name')          else 'unknown'
    score = m.GetProp('GNINA_CNN_score') if m.HasProp('GNINA_CNN_score') else 'N/A'
    print(f'  {name:20s}  CNN={score}')

✓ GNINA hits SDF: phase3_step2_hits.sdf (9 KB)
✓ Protein PDB: synthetic_holo_template.pdb (166 KB)
✓ GROMACS: found

GNINA hit molecules: 5
  candidate_0004        CNN=0.7567
  candidate_0001        CNN=0.7177
  candidate_0002        CNN=0.6376
  candidate_0003        CNN=0.6055
  candidate_0005        CNN=0.4383


---
# STEP 10 · Protein Topology
Convert `synthetic_holo_template.pdb` → GROMACS format via `pdb2gmx`.  
Uses AMBER99sb-ildn + TIP3P (confirmed available on Colab GROMACS 2021).  
Auto-assigns histidine protonation. Total charge = -6e (neutralized in Step 13).

In [5]:
# Remove HETATM records — pdb2gmx needs protein-only PDB
protein_clean = WORK_DIR / 'protein_clean.pdb'
with open(PROTEIN_PDB) as f:
    lines = f.readlines()
clean = [l for l in lines if l.startswith(('ATOM','TER','END'))]
with open(protein_clean, 'w') as f:
    f.writelines(clean)
print(f'✓ Cleaned PDB: {len(clean)} lines')

# Run pdb2gmx
protein_gro = WORK_DIR / 'protein.gro'
protein_top = WORK_DIR / 'topol.top'

r = subprocess.run([
    'gmx', 'pdb2gmx',
    '-f',  str(protein_clean),
    '-o',  str(protein_gro),
    '-p',  str(protein_top),
    '-i',  str(WORK_DIR / 'posre.itp'),
    '-ff', FORCE_FIELD,
    '-water', WATER_MODEL,
    '-ignh',
], capture_output=True, text=True, input='1\n' * 20)

if protein_gro.exists():
    print(f'\n✓ STEP 10 COMPLETE')
    print(f'  protein.gro : {protein_gro.stat().st_size/1024:.1f} KB')
    print(f'  topol.top   : {protein_top.stat().st_size/1024:.1f} KB')
    for line in r.stdout.split('\n'):
        if any(k in line for k in ['Total mass','Total charge','dihedrals','successfully generated']):
            print(f'  {line.strip()}')
else:
    print('✗ pdb2gmx failed:')
    print(r.stderr[-1500:])
    raise RuntimeError('Step 10 failed')

✓ Cleaned PDB: 2073 lines

✓ STEP 10 COMPLETE
  protein.gro : 180.9 KB
  topol.top   : 1134.0 KB
  Generating angles, dihedrals and pairs...
  There are 11413 dihedrals,  826 impropers, 7530 angles
  Total mass 29535.114 a.m.u.
  Total charge -6.000 e
  You have successfully generated a topology from: /content/gromacs_md/protein_clean.pdb.


---
# STEP 11 · Ligand Parameterization
Extract best GNINA docked pose from `phase3_step2_hits.sdf`.  
Save ligand coordinates as SDF and GRO for reference.  

> **Note:** Full GAFF2 parameterization requires AmberTools (`antechamber`) which
> is not available on Colab. For the Colab validation run, we save the ligand
> geometry and proceed with protein-only solvation (Steps 12–13).  
> For the real MD run, use: `acpype -i ligand.sdf -c bcc -f GAFF2` in a local conda env.

In [6]:
# Extract best candidate
best_mol = None
for mol in hit_mols:
    if mol.GetProp('_Name') if mol.HasProp('_Name') else '' == BEST_CANDIDATE:
        best_mol = mol
        break
if best_mol is None:
    best_mol = hit_mols[0]
    BEST_CANDIDATE = best_mol.GetProp('_Name') if best_mol.HasProp('_Name') else 'ligand'

safe_name = BEST_CANDIDATE.replace('-','_')
lig_dir   = WORK_DIR / safe_name
lig_dir.mkdir(exist_ok=True)

score = best_mol.GetProp('GNINA_CNN_score') if best_mol.HasProp('GNINA_CNN_score') else 'N/A'
vina  = best_mol.GetProp('GNINA_Vina_aff')  if best_mol.HasProp('GNINA_Vina_aff')  else 'N/A'
print(f'Ligand          : {BEST_CANDIDATE}')
print(f'GNINA CNN score : {score}')
print(f'Vina affinity   : {vina} kcal/mol')

# Save ligand SDF (with Hs)
mol_h = Chem.AddHs(best_mol)
if mol_h.GetNumConformers() == 0:
    AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3())
    AllChem.MMFFOptimizeMolecule(mol_h)

lig_sdf = lig_dir / f'{safe_name}.sdf'
w = SDWriter(str(lig_sdf))
w.write(mol_h)
w.close()

# Save ligand GRO (coordinates only — no force field params)
lig_gro = lig_dir / f'{safe_name}.gro'
conf = mol_h.GetConformer()
with open(lig_gro, 'w') as f:
    f.write(f'{safe_name} GNINA docked pose\n')
    f.write(f'{mol_h.GetNumAtoms()}\n')
    for i, atom in enumerate(mol_h.GetAtoms()):
        pos = conf.GetAtomPosition(i)
        sym = atom.GetSymbol()
        f.write(f'{1:5d}{safe_name[:4]:5s}{sym+str(i+1):5s}{i+1:5d}'
                f'{pos.x/10:8.3f}{pos.y/10:8.3f}{pos.z/10:8.3f}\n')
    f.write('   5.0   5.0   5.0\n')

print(f'\n✓ STEP 11 COMPLETE')
print(f'  {lig_sdf.name} : {lig_sdf.stat().st_size} bytes (GNINA pose)')
print(f'  {lig_gro.name} : {lig_gro.stat().st_size} bytes (coordinates)')
print(f'\n  ⚠ For GAFF2 ITP: run locally with:')
print(f'    conda install -c conda-forge ambertools')
print(f'    acpype -i {safe_name}.sdf -c bcc -f GAFF2 -o gmx')

Ligand          : candidate_0004
GNINA CNN score : 0.7567
Vina affinity   : 3.08 kcal/mol

✓ STEP 11 COMPLETE
  candidate_0004.sdf : 1871 bytes (GNINA pose)
  candidate_0004.gro : 775 bytes (coordinates)

  ⚠ For GAFF2 ITP: run locally with:
    conda install -c conda-forge ambertools
    acpype -i candidate_0004.sdf -c bcc -f GAFF2 -o gmx


---
# STEP 12 · Complex Assembly (Coordinates)
Merge protein + ligand GRO files into `complex.gro`.  
The topology (`complex.top`) uses protein-only parameters for now.  
When GAFF2 ITP is available, add `#include` statement and rerun Steps 13 onward.

In [7]:
complex_gro = lig_dir / 'complex.gro'
complex_top = lig_dir / 'complex.top'

# Merge GRO files
prot_lines = open(protein_gro).readlines()
lig_lines  = open(lig_gro).readlines()
prot_atoms = prot_lines[2:-1]
lig_atoms  = lig_lines[2:-1]
total_atoms = len(prot_atoms) + len(lig_atoms)

with open(complex_gro, 'w') as f:
    f.write(f'INSR + {safe_name} complex\n')
    f.write(f'{total_atoms}\n')
    f.writelines(prot_atoms)
    f.writelines(lig_atoms)
    f.write(prot_lines[-1])  # box dimensions from protein

# Copy protein topology (ligand ITP will be added when GAFF2 available)
shutil.copy(protein_top, complex_top)

print(f'✓ STEP 12 COMPLETE')
print(f'  Protein atoms : {len(prot_atoms)}')
print(f'  Ligand atoms  : {len(lig_atoms)}')
print(f'  Total atoms   : {total_atoms}')
print(f'  complex.gro   : {complex_gro.stat().st_size/1024:.1f} KB')
print(f'  complex.top   : {complex_top.stat().st_size/1024:.1f} KB (protein-only)')

✓ STEP 12 COMPLETE
  Protein atoms : 4114
  Ligand atoms  : 16
  Total atoms   : 4130
  complex.gro   : 181.6 KB
  complex.top   : 1134.0 KB (protein-only)


---
# STEP 13 · Solvation
Runs on the **protein structure** (dodecahedron, TIP3P, 0.15M NaCl).  
This validates the full GROMACS solvation pipeline end-to-end.  
When GAFF2 ligand ITP is available, re-run with `complex.top` including ligand.

**Pipeline spec:**
- Dodecahedron box, minimum 12Å from solute to edge
- TIP3P water model (`gmx solvate`)
- Na⁺/Cl⁻ to neutralize and reach 0.15M NaCl (`gmx genion`)

In [8]:
box_gro      = WORK_DIR / 'box.gro'
solvated_gro = WORK_DIR / 'solvated.gro'
ions_tpr     = WORK_DIR / 'ions.tpr'
system_gro   = WORK_DIR / 'system.gro'
ions_mdp     = WORK_DIR / 'ions.mdp'

# Use protein-only topology for solvation (avoids GAFF2 ITP issue)
solvation_top = protein_top

# 13a: Dodecahedron box — 12Å padding
print('13a: Creating dodecahedron box (12Å)...')
r = subprocess.run([
    'gmx','editconf',
    '-f', str(protein_gro),   # protein only
    '-o', str(box_gro),
    '-bt','dodecahedron',
    '-d', str(BOX_DISTANCE),
], capture_output=True, text=True)
if not box_gro.exists():
    raise RuntimeError(f'editconf failed: {r.stderr[-300:]}')
print(f'  ✓ box.gro created')

# 13b: Solvate with TIP3P water
print('13b: Solvating with TIP3P water...')
r = subprocess.run([
    'gmx','solvate',
    '-cp', str(box_gro),
    '-cs', 'spc216.gro',
    '-o',  str(solvated_gro),
    '-p',  str(solvation_top),
], capture_output=True, text=True)
if not solvated_gro.exists():
    raise RuntimeError(f'solvate failed: {r.stderr[-300:]}')
# Count water molecules added
for line in r.stderr.split('\n'):
    if 'SOL' in line and 'molecules' in line.lower():
        print(f'  ✓ {line.strip()}')
        break
print(f'  ✓ solvated.gro created')

# 13c: Prepare ions.mdp for grompp
ions_mdp.write_text(
    'integrator = steep\n'
    'nsteps     = 0\n'
    'emtol      = 1000\n'
    'emstep     = 0.01\n'
)
r = subprocess.run([
    'gmx','grompp',
    '-f', str(ions_mdp),
    '-c', str(solvated_gro),
    '-p', str(solvation_top),
    '-o', str(ions_tpr),
    '-maxwarn','5',
], capture_output=True, text=True)
if not ions_tpr.exists():
    raise RuntimeError(f'grompp failed: {r.stderr[-500:]}')
print('  ✓ ions.tpr ready')

# 13d: Add Na+/Cl- ions
print(f'13c: Adding Na+/Cl- (neutral + {IONIC_STRENGTH}M NaCl)...')
r = subprocess.run([
    'gmx','genion',
    '-s',  str(ions_tpr),
    '-o',  str(system_gro),
    '-p',  str(solvation_top),
    '-pname','NA','-nname','CL',
    '-neutral',
    '-conc', str(IONIC_STRENGTH),
], capture_output=True, text=True, input='SOL\n')

if not system_gro.exists():
    raise RuntimeError(f'genion failed: {r.stderr[-300:]}')

n_atoms = int(open(system_gro).readlines()[1].strip())
# Count ions added
na_count = cl_count = 0
for line in r.stderr.split('\n'):
    if 'NA' in line and 'added' in line.lower(): na_count = line
    if 'CL' in line and 'added' in line.lower(): cl_count = line

print(f'\n✓ STEP 13 COMPLETE')
print(f'  system.gro     : {system_gro.stat().st_size/1024:.1f} KB')
print(f'  Total atoms    : {n_atoms}')
print(f'  System contains: INSR protein + TIP3P water + Na/Cl ions')

13a: Creating dodecahedron box (12Å)...
  ✓ box.gro created
13b: Solvating with TIP3P water...
  ✓ solvated.gro created
  ✓ ions.tpr ready
13c: Adding Na+/Cl- (neutral + 0.15M NaCl)...

✓ STEP 13 COMPLETE
  system.gro     : 1801.3 KB
  Total atoms    : 40988
  System contains: INSR protein + TIP3P water + Na/Cl ions


---
## Summary

In [9]:
print('=' * 60)
print('  PHASE 3.3 STEP 1 — SYSTEM PREPARATION COMPLETE')
print('=' * 60)
print(f'  Step 10 — Protein topology   : ✓ protein.gro + topol.top')
print(f'  Step 11 — Ligand coordinates  : ✓ {safe_name}.sdf + .gro (coords only)')
print(f'  Step 12 — Complex assembly    : ✓ complex.gro ({total_atoms} atoms)')
print(f'  Step 13 — Solvation           : ✓ system.gro ({n_atoms} atoms)')
print('=' * 60)
print()
print('  ⚠ Ligand GAFF2 ITP (Step 11) — requires local AmberTools:')
print('    conda install -c conda-forge ambertools acpype')
print('    acpype -i candidate_0004.sdf -c bcc -f GAFF2 -o gmx')
print('    Then add #include to complex.top and re-run Step 13')
print()
print('  Next: Phase 3.3 Step 2 — Energy Minimization')

  PHASE 3.3 STEP 1 — SYSTEM PREPARATION COMPLETE
  Step 10 — Protein topology   : ✓ protein.gro + topol.top
  Step 11 — Ligand coordinates  : ✓ candidate_0004.sdf + .gro (coords only)
  Step 12 — Complex assembly    : ✓ complex.gro (4130 atoms)
  Step 13 — Solvation           : ✓ system.gro (40988 atoms)

  ⚠ Ligand GAFF2 ITP (Step 11) — requires local AmberTools:
    conda install -c conda-forge ambertools acpype
    acpype -i candidate_0004.sdf -c bcc -f GAFF2 -o gmx
    Then add #include to complex.top and re-run Step 13

  Next: Phase 3.3 Step 2 — Energy Minimization


---
## Download All GROMACS Files

In [10]:
from google.colab import files

zip_path = f'/content/gromacs_system_{safe_name}.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in [
        protein_gro, protein_top,          # Step 10
        lig_sdf, lig_gro,                  # Step 11
        complex_gro, complex_top,          # Step 12
        box_gro, solvated_gro, system_gro, # Step 13
        ions_mdp,
    ]:
        if Path(f).exists():
            zf.write(f, Path(f).name)

files.download(zip_path)
print(f'✓ Downloaded: {Path(zip_path).name}')
print(f'  Contents: protein.gro, topol.top, complex.gro, system.gro + ligand files')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded: gromacs_system_candidate_0004.zip
  Contents: protein.gro, topol.top, complex.gro, system.gro + ligand files


In [11]:
import zipfile

zip_path = f'/content/gromacs_system_candidate_0004.zip'
with zipfile.ZipFile(zip_path, 'r') as zf:
    print("Files inside zip:")
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f"  {name:40s} {info.file_size/1024:.1f} KB")

Files inside zip:
  protein.gro                              180.9 KB
  topol.top                                1134.0 KB
  candidate_0004.sdf                       1.8 KB
  candidate_0004.gro                       0.8 KB
  complex.gro                              181.6 KB
  complex.top                              1134.0 KB
  box.gro                                  180.9 KB
  solvated.gro                             1808.6 KB
  system.gro                               1801.3 KB
  ions.mdp                                 0.1 KB


In [12]:
import zipfile
from pathlib import Path

zip_path = '/content/gromacs_system_candidate_0004.zip'
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extract('system.gro', '/content/')

with open('/content/system.gro', 'r') as f:
    f.readline()
    n_atoms = int(f.readline().strip())

print(f"n_atoms = {n_atoms}")

n_atoms = 40988


---
## Next → Phase 3.3 Step 2: Energy Minimization

| Step | Task | Command |
|---|---|---|
| 14 | Steepest descent (50,000 steps) | `gmx mdrun -deffnm em` |
| 15 | Check Fmax < 1000 kJ/mol/nm | `gmx energy -f em.edr` |
| 16 | NVT equilibration 100ps 310K | `gmx mdrun -deffnm nvt` |
| 17 | NPT equilibration 100ps 1 bar | `gmx mdrun -deffnm npt` |
| 18 | 100ns production MD | `gmx mdrun -deffnm md` |